# 0. Imports

In [ ]:
import numpy as np
import pandas as pd


import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor


from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', 50)
PAX_TARGET = "OD_PAX_LOG"  
REV_TARGET = "OD_REV_USD_LOG"      
VAL_MONTHS = 12             


# 1. Organizing

In [ ]:
full_spine = pd.read_csv('./data/made/AA_MASTER_V10.csv')

In [ ]:
full_spine.info()

In [ ]:
full_spine.drop(columns=['ORG_CARGO_SL','ORG_CARGO_HUB','DES_CARGO_SL','DES_CARGO_HUB','ORG_CNTRY_CD',
                         'DES_CNTRY_CD'],inplace=True)
full_spine.describe()


In [ ]:
full_spine.to_csv('./data/made/AA_MASTER_V10.csv',index=False)

In [ ]:
def prepare_datasets(
    df: pd.DataFrame,
    target_col: str,
    val_months: int = 12,
    drop_covid_targets: bool = True,
):
    """
    Separates `full_spine` into leak-free train, validation, and 2027-2028 forecast sets.
    Automatically casts object/string features to `category` dtype for native tree support.
    """
    work_df = df.copy()

    # 1. Define columns that dont enter X 
    non_feature_cols = {
        "ROUTE", "ORG_AIRPRT", "DES_AIRPRT", "DEP_YEAR", "DEP_MONTH",
        PAX_TARGET, REV_TARGET, "COVID",
    }
    feature_cols = [c for c in work_df.columns if c not in non_feature_cols]

    # 2. Cast remaining object/string columns to 'category'
    cat_cols = work_df[feature_cols].select_dtypes(include=["object", "string"]).columns.tolist()
    for col in cat_cols:
        work_df[col] = work_df[col].astype("category")

    # 3. Create helper chronological timestamp for exact time-based slicing
    work_df["_ts"] = pd.to_datetime(
        dict(year=work_df["DEP_YEAR"], month=work_df["DEP_MONTH"], day=1)
    )

    # Auto-detect exact historical cutoff T 
    cutoff_t = work_df.loc[work_df[target_col].notna(), "_ts"].max()
    val_start = cutoff_t - pd.DateOffset(months=val_months - 1)

    print(f"Historical Cutoff T: {cutoff_t.strftime('%Y-%m')}")
    print(f"Validation Window:   {val_start.strftime('%Y-%m')} -> {cutoff_t.strftime('%Y-%m')}")

    # 4. Filter valid historical rows 
    hist_mask = work_df[target_col].notna() & work_df[f"{target_col}_L24"].notna()
    if drop_covid_targets:
        hist_mask = hist_mask & (~work_df["DEP_YEAR"].isin([2020, 2021]))

    train_df = work_df[hist_mask & (work_df["_ts"] < val_start)].copy()
    val_df = work_df[hist_mask & (work_df["_ts"] >= val_start)].copy()
    forecast_df = work_df[work_df["_ts"] > cutoff_t].copy()

    return train_df, val_df, forecast_df, feature_cols, cat_cols


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

def get_tree_models(cat_cols: list, num_cols: list) -> dict:
    """Returns native-NaN gradient boosters + a NaN-safe Random Forest pipeline."""
    
    # Preprocessing for RandomForest 
    rf_preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="constant", fill_value=-999.0), num_cols),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-2), cat_cols),
        ]
    )

    rf_pipeline = Pipeline([
        ("prep", rf_preprocessor),
        ("rf", RandomForestRegressor(
            n_estimators=150,       
            max_depth=16,           
            min_samples_leaf=10,
            max_samples=0.5,        
            random_state=42,
            n_jobs=-1,
        )),
    ])

    return {
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=500, learning_rate=0.05, num_leaves=63,
            subsample_for_bin=200_000, random_state=42, n_jobs=-1,
        ),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=7,
            tree_method="hist", enable_categorical=True, random_state=42, n_jobs=-1,
        ),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=500, learning_rate=0.05, max_leaf_nodes=63,
            categorical_features="from_dtype", random_state=42,
        ),
        "RandomForest": rf_pipeline,
    }


def evaluate_predictions(y_true_log1p: pd.Series, y_pred_log1p: np.ndarray) -> dict:
    """Evaluates error in BOTH log1p space (model scale) and raw expm1 space (real units)."""
   
    y_pred_log1p = np.clip(y_pred_log1p, 0.0, None)

    y_true_raw = np.expm1(y_true_log1p)
    y_pred_raw = np.expm1(y_pred_log1p)

    return {
        "RMSE_log1p": round(root_mean_squared_error(y_true_log1p, y_pred_log1p), 4),
        "R2_log1p": round(r2_score(y_true_log1p, y_pred_log1p), 4),
        "MAE_raw": round(mean_absolute_error(y_true_raw, y_pred_raw), 2),
        "RMSE_raw": round(root_mean_squared_error(y_true_raw, y_pred_raw), 2),
    }


In [ ]:

train_df, val_df, forecast_df, feature_cols, cat_cols = prepare_datasets(
    df=full_spine,
    target_col=PAX_TARGET,
    val_months=VAL_MONTHS,
    drop_covid_targets=True,
)

In [ ]:

X_train, y_train = train_df[feature_cols], train_df[PAX_TARGET]
X_val, y_val = val_df[feature_cols], val_df[PAX_TARGET]

print(f"Train rows: {len(X_train):,} | Val rows: {len(X_val):,} | Features: {len(feature_cols)}")

# Train & benchmark each tree model
models = get_tree_models(cat_cols, num_cols)
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")  
    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)

    metrics = evaluate_predictions(y_val, val_preds)
    metrics["Model"] = name
    results.append(metrics)

# 3. Compare validation scores side-by-side
results_df = pd.DataFrame(results)[["Model", "RMSE_log1p", "R2_log1p", "MAE_raw", "RMSE_raw"]]
print("\n--- Validation Benchmark ---")
print(results_df.to_string(index=False))


In [ ]:

train_df, val_df, forecast_df, feature_cols, cat_cols = prepare_datasets(
    df=full_spine,
    target_col=REV_TARGET,
    val_months=VAL_MONTHS,
    drop_covid_targets=True,
)

In [ ]:

X_train, y_train = train_df[feature_cols], train_df[REV_TARGET]
X_val, y_val = val_df[feature_cols], val_df[REV_TARGET]

print(f"Train rows: {len(X_train):,} | Val rows: {len(X_val):,} | Features: {len(feature_cols)}")

# Train & benchmark each tree model
models = get_tree_models(cat_cols, num_cols)
results = []

for name, model in models.items():
    print(f"\nTraining {name}...")  
    model.fit(X_train, y_train)
    val_preds = model.predict(X_val)

    metrics = evaluate_predictions(y_val, val_preds)
    metrics["Model"] = name
    results.append(metrics)

# Compare validation scores side-by-side
results_df = pd.DataFrame(results)[["Model", "RMSE_log1p", "R2_log1p", "MAE_raw", "RMSE_raw"]]
print("\n--- Validation Benchmark ---")
print(results_df.to_string(index=False))


In [ ]:
import json
import joblib

full_spine.to_parquet("full_spine_ready.parquet", index=False)



# Save feature lists & ensemble weight as plain JSON
metadata = {
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
}
with open("airline_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)


